# FPL Decision Support System — Modelling Notebook
**Thesis Project** | PyCaret AutoML vs AutoKeras Deep Learning

---
### ⚠️ Before running anything:
1. Go to `Runtime` → `Change runtime type` → select **T4 GPU** → Save
2. Run cells **in order** and follow the restart instruction in Section 1

### Pipeline
- **Section 1** — Mount Drive → Install PyCaret → Restart → Install AutoKeras
- **Section 2** — Load & Prepare Data
- **Section 3** — PyCaret AutoML Benchmarking
- **Section 4** — AutoKeras Deep Learning
- **Section 5** — Model Comparison
- **Section 6** — Prediction Intervals
- **Section 7** — Save Predictions for Dashboard

---
## Section 1 — Setup
### Step 1a: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

### Step 1b: Find your CSV path
Run this to print the exact path of your CSVs in Drive. Copy the `featured_training_set.csv` path into `DATA_PATH` in Section 2.

In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

### Step 1c: Install PyCaret
⚠️ **After this cell finishes — restart the runtime before continuing.**

`Runtime` → `Restart runtime` → then run **Step 1d** next.

In [ ]:
!pip install pycaret[full] -q
print('PyCaret installed. RESTART THE RUNTIME NOW: Runtime → Restart runtime')

### Step 1d: After restarting — remount Drive & install AutoKeras
Run this as your first cell after the restart.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install autokeras -q

print('Setup complete. Ready to run the notebook.')

---
## Section 2 — Load & Prepare Data

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paste the path printed in Step 1b here ────────────────────────────────────
DATA_PATH = '/content/drive/MyDrive/data/processed/featured_training_set.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# ── Create the TARGET: next GW points for each player ─────────────────────────
# Shift total_points by -1 within each player+season group.
# Given THIS gameweek's features → predict NEXT gameweek's points.
df = df.sort_values(['name', 'season', 'GW'])
df['target'] = df.groupby(['name', 'season'])['total_points'].shift(-1)

# Drop the last GW of each season (no next GW to predict)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f'After creating target: {df.shape[0]} rows')

In [ ]:
# ── Drop identifier and leakage columns ───────────────────────────────────────
DROP_COLS = [
    'name', 'team', 'season', 'GW', 'kickoff_time',
    'total_points',  # raw target — already encoded as 'target'
    'element',       # player ID
    'fixture',       # fixture ID
    'modified',      # metadata
]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# ── Encode categoricals ────────────────────────────────────────────────────────
df['position'] = df['position'].astype('category').cat.codes  # GK=0 DEF=1 MID=2 FWD=3
df['was_home'] = df['was_home'].astype(int)

# ── Build modelling dataframe ──────────────────────────────────────────────────
model_df = df.drop(columns=DROP_COLS)

# Drop any remaining non-numeric columns
non_numeric = model_df.select_dtypes(exclude=[np.number, 'bool']).columns.tolist()
if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    model_df = model_df.drop(columns=non_numeric)

print(f'Modelling dataframe: {model_df.shape}')
print(f'Target stats:\n{model_df["target"].describe()}')

In [ ]:
# ── Walk-Forward Train / Test Split ───────────────────────────────────────────
# Add season and GW back temporarily just for splitting
model_df['season'] = df['season'].values
model_df['GW']     = df['GW'].values

train_mask = (
    model_df['season'].isin(['2324', '2425']) |
    ((model_df['season'] == '2526') & (model_df['GW'] <= 25))
)
test_mask = (model_df['season'] == '2526') & (model_df['GW'] >= 26)

train_df  = model_df[train_mask].drop(columns=['season', 'GW']).reset_index(drop=True)
test_df   = model_df[test_mask].drop(columns=['season', 'GW']).reset_index(drop=True)

# Save player metadata for the final predictions output
test_meta = df[test_mask][['name', 'team', 'position', 'GW', 'season', 'value']].reset_index(drop=True)

print(f'Train: {train_df.shape} | Test: {test_df.shape}')

---
## Section 3 — PyCaret AutoML Benchmarking
PyCaret automatically trains and compares ~20 regression models using 5-fold cross-validation.

In [ ]:
from pycaret.regression import *

pc_setup = setup(
    data            = train_df,
    target          = 'target',
    session_id      = 42,
    fold            = 5,
    verbose         = True,
    use_gpu         = True,
    remove_outliers = True,
    normalize       = True,
)
print('PyCaret setup complete.')

In [ ]:
# Benchmark all models — sorted by MAE
# n_select=3 keeps the top 3 for the ensemble prediction interval later
best_models = compare_models(
    sort     = 'MAE',
    n_select = 3,
    exclude  = ['ransac']
)
print('\nTop 3 models selected.')

In [ ]:
# Tune the best model
tuned_model = tune_model(best_models[0], optimize='MAE', n_iter=20)
print(f'Best model after tuning: {type(tuned_model).__name__}')

In [ ]:
# Finalise — retrain on full training data
from sklearn.metrics import mean_absolute_error, r2_score

final_pycaret_model = finalize_model(tuned_model)
pycaret_preds = predict_model(final_pycaret_model, data=test_df)
mae_pc = mean_absolute_error(pycaret_preds['target'], pycaret_preds['prediction_label'])
r2_pc  = r2_score(pycaret_preds['target'], pycaret_preds['prediction_label'])

print(f'PyCaret Test MAE : {mae_pc:.4f}')
print(f'PyCaret Test R²  : {r2_pc:.4f}')

In [ ]:
os.makedirs('/content/drive/MyDrive/fpl_thesis/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/fpl_thesis/outputs', exist_ok=True)

save_model(final_pycaret_model, '/content/drive/MyDrive/fpl_thesis/models/pycaret_best_model')
print('PyCaret model saved.')

---
## Section 4 — AutoKeras Deep Learning
AutoKeras automatically searches for the best neural network architecture for your data.

In [ ]:
import autokeras as ak
import tensorflow as tf

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

X_train = train_df.drop(columns=['target']).values.astype(np.float32)
y_train = train_df['target'].values.astype(np.float32)
X_test  = test_df.drop(columns=['target']).values.astype(np.float32)
y_test  = test_df['target'].values.astype(np.float32)

In [ ]:
# AutoKeras tries 10 neural architectures and picks the best
ak_model = ak.StructuredDataRegressor(
    max_trials  = 10,
    overwrite   = True,
    seed        = 42,
    objective   = 'val_mean_absolute_error',
    directory   = '/content/drive/MyDrive/fpl_thesis/models/autokeras_trials'
)

ak_model.fit(
    X_train, y_train,
    epochs           = 50,
    validation_split = 0.1,
    verbose          = 1
)
print('AutoKeras training complete.')

In [ ]:
ak_preds_raw = ak_model.predict(X_test).flatten()
mae_ak = mean_absolute_error(y_test, ak_preds_raw)
r2_ak  = r2_score(y_test, ak_preds_raw)

print(f'AutoKeras Test MAE : {mae_ak:.4f}')
print(f'AutoKeras Test R²  : {r2_ak:.4f}')

best_ak_model = ak_model.export_model()
best_ak_model.save('/content/drive/MyDrive/fpl_thesis/models/autokeras_best_model.keras')
print('AutoKeras model saved.')

---
## Section 5 — Model Comparison

In [ ]:
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    'Model' : [f'PyCaret ({type(tuned_model).__name__})', 'AutoKeras (Neural Network)'],
    'MAE'   : [round(mae_pc, 4), round(mae_ak, 4)],
    'R²'    : [round(r2_pc,  4), round(r2_ak,  4)],
})

print('===== Model Comparison =====')
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['steelblue', 'darkorange']

comparison.plot(kind='bar', x='Model', y='MAE', ax=axes[0], color=colors, legend=False)
axes[0].set_title('MAE — lower is better')
axes[0].set_xticklabels(comparison['Model'], rotation=15, ha='right')

comparison.plot(kind='bar', x='Model', y='R²', ax=axes[1], color=colors, legend=False)
axes[1].set_title('R² — higher is better')
axes[1].set_xticklabels(comparison['Model'], rotation=15, ha='right')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/fpl_thesis/outputs/model_comparison.png', dpi=150)
plt.show()

winner = 'PyCaret' if mae_pc <= mae_ak else 'AutoKeras'
print(f'\nWinner: {winner} (lower MAE)')

---
## Section 6 — Prediction Intervals (Low / Mid / High)
Combines predictions from all 4 models (3 PyCaret + AutoKeras):
- **Mid** = mean across all models
- **Low** = mid − 1 std (pessimistic)
- **High** = mid + 1 std (optimistic)
- **Confidence** = bucketed from interval width (narrow = High confidence)

In [ ]:
all_preds = []

# Collect from top 3 PyCaret models
for i, m in enumerate(best_models):
    fm = finalize_model(m)
    p  = predict_model(fm, data=test_df)['prediction_label'].values
    all_preds.append(p)
    print(f'Collected PyCaret model {i+1}')

# Add AutoKeras
all_preds.append(ak_preds_raw)
print('Collected AutoKeras')

# Stack: shape (4 models, n_test_rows)
preds_array = np.array(all_preds)
mid  = preds_array.mean(axis=0)
std  = preds_array.std(axis=0)
low  = np.clip(mid - std, 0, None)  # FPL points can't be negative
high = mid + std

# Confidence bucketing
interval_width = high - low
q33, q66 = np.percentile(interval_width, [33, 66])

def get_confidence(w):
    if w <= q33:   return 'High'
    elif w <= q66: return 'Medium'
    else:          return 'Low'

confidence = [get_confidence(w) for w in interval_width]
print(f'\nIntervals computed for {len(mid)} rows')

---
## Section 7 — Save Final Predictions for Dashboard

In [ ]:
predictions_df = test_meta.copy()
predictions_df['predicted_pts_mid']  = np.round(mid,  2)
predictions_df['predicted_pts_low']  = np.round(low,  2)
predictions_df['predicted_pts_high'] = np.round(high, 2)
predictions_df['confidence']         = confidence
predictions_df['actual_pts']         = y_test

print(predictions_df[[
    'name', 'team', 'position', 'GW', 'value',
    'predicted_pts_low', 'predicted_pts_mid',
    'predicted_pts_high', 'confidence', 'actual_pts'
]].head(10))

OUT_PATH = '/content/drive/MyDrive/fpl_thesis/data/processed/predictions.csv'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
predictions_df.to_csv(OUT_PATH, index=False)

print(f'\nSaved to {OUT_PATH}')
print(f'Shape: {predictions_df.shape}')

---
### ✅ Notebook Complete

Files saved to Google Drive:
- `models/pycaret_best_model` — best traditional ML model
- `models/autokeras_best_model.keras` — best deep learning model
- `outputs/model_comparison.png` — comparison chart for your thesis
- `data/processed/predictions.csv` — predictions ready for PuLP + dashboard

**Next step → PuLP optimisation: squad selection from these predictions.**